# Detector study on Kaggle (GPU)

**Settings (right panel):** Accelerator = **GPU T4 x2** (or P100), Internet = **On**.

**Secret:** Add-ons → Secrets → add `HF_TOKEN` = your Hugging Face read token (IndicDLP needs a login). Tick it for this notebook.

**Input:** a Kaggle dataset with `train_detector.py`, `build_subset.py`, `fix_labels.py`, `indic.yaml`, `yolov10x_indicdlp.pt`, `yolov8n-doclaynet.pt`.

**PART = 1**: D0-D6 (transfer study). **PART = 2**: A1-A4 (augmentation study, on the best transfer setup).
Run with **Save Version → Save & Run All (Commit)**. Results: Output tab → `runs_det_part1.zip` / `runs_det_part2.zip`.

In [ ]:
PART = 1          # 1 = transfer study, 2 = augmentation study
AUG_BASE = 'yolov8n-doclaynet.pt'   # PART 2 starts from this; change to 'yolov8n.pt' if D2 beat D4 in PART 1

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install (Kaggle already has PyTorch)
!pip install -q ultralytics datasets

In [ ]:
# 3. Copy the uploaded scripts and weights
import glob, os, shutil
os.makedirs('/tmp/build', exist_ok=True)
for name in ['train_detector.py', 'yolov10x_indicdlp.pt', 'yolov8n-doclaynet.pt']:
    found = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert found, f'{name} not found in /kaggle/input - add your dataset with Add Input'
    shutil.copy(found[0], f'/kaggle/working/{name}')
for name in ['build_subset.py', 'fix_labels.py', 'indic.yaml']:
    found = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert found, f'{name} not found in /kaggle/input'
    shutil.copy(found[0], f'/tmp/build/{name}')
os.chdir('/kaggle/working')
print(os.listdir('/kaggle/working'), os.listdir('/tmp/build'))

In [ ]:
# 4. Build the same 4000 / 500 / 500 subset from Hugging Face (about 15-30 min), in /tmp so it is not in the Output
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
if not os.path.exists('/tmp/build/dataset_root/labels/test'):
    !cd /tmp/build && python build_subset.py
!cd /tmp/build && python fix_labels.py
print('expected checksum: 8bc1224593761ceeba440ac23dd432d1   (same = exactly the data used on the laptop)')

In [ ]:
# 5. Data file with absolute paths
yaml_text = open('/tmp/build/indic.yaml').read().split('names:')[1]
with open('/kaggle/working/data_kaggle.yaml', 'w') as f:
    f.write('path: /tmp/build/dataset_root
train: images/train
val: images/val
test: images/test

names:' + yaml_text)
print(open('/kaggle/working/data_kaggle.yaml').read()[:300])
BASE = 'python train_detector.py --data data_kaggle.yaml --device 0 --workers 4'

## PART 1: transfer study (all YOLOv8n unless noted, 40 epochs, early stopping patience 10)

In [ ]:
# D0: released YOLOv10x IndicDLP checkpoint, test only (reference; labels now fixed)
if PART == 1:
    !{BASE} --name D0_released --model yolov10x_indicdlp.pt --eval-only

In [ ]:
# D1: from scratch (random weights)
if PART == 1:
    !{BASE} --name D1_scratch --model yolov8n.yaml

In [ ]:
# D2: COCO pre-trained, full fine-tuning
if PART == 1:
    !{BASE} --name D2_coco_full --model yolov8n.pt

In [ ]:
# D3: COCO pre-trained, backbone frozen (layers 0-9)
if PART == 1:
    !{BASE} --name D3_coco_frozen --model yolov8n.pt --freeze 10

In [ ]:
# D4: DocLayNet pre-trained (documents), full fine-tuning
if PART == 1:
    !{BASE} --name D4_doclaynet_full --model yolov8n-doclaynet.pt

In [ ]:
# D5: DocLayNet pre-trained, backbone frozen
if PART == 1:
    !{BASE} --name D5_doclaynet_frozen --model yolov8n-doclaynet.pt --freeze 10

In [ ]:
# D6: bigger model (YOLOv8s), COCO pre-trained, full fine-tuning
if PART == 1:
    !{BASE} --name D6_v8s_coco_full --model yolov8s.pt

## PART 2: augmentation study (the 'default' run is D4 or D2 from PART 1)

In [ ]:
if PART == 2:
    !{BASE} --name A1_none --model {AUG_BASE} --aug none

In [ ]:
if PART == 2:
    !{BASE} --name A2_noflip --model {AUG_BASE} --aug noflip

In [ ]:
if PART == 2:
    !{BASE} --name A3_nomosaic --model {AUG_BASE} --aug nomosaic

In [ ]:
if PART == 2:
    !{BASE} --name A4_doc --model {AUG_BASE} --aug doc

In [ ]:
# Results, then one zip to download (Output tab). Model weights are kept (small for YOLOv8n/s).
import pandas as pd
print(pd.read_csv('runs_det/summary.csv').to_string())
for f in ['yolov10x_indicdlp.pt', 'yolov8n-doclaynet.pt', 'yolov8n.pt', 'yolov8s.pt']:
    if os.path.exists(f): os.remove(f)   # keep the Output small
shutil.make_archive(f'/kaggle/working/runs_det_part{PART}', 'zip', '/kaggle/working', 'runs_det')